# 04 · Análise dos resultados

Lê os arquivos da avaliação e monta a comparação dos 3 sistemas: a tabela com todas as métricas, o gráfico de acerto × custo, os erros por pergunta, o roteamento, o verificador e a concordância entre o juiz, a revisão humana e a checagem automática. **Só lê arquivos**: roda no Colab com o runtime padrão (só CPU) ou localmente.

| Arquivo (em `resultados/`) | De onde vem |
|---|---|
| `execucoes.csv`, `execucoes.jsonl` | notebook 03 (as 60 execuções) |
| `juiz.csv`, `juiz.jsonl` | `avaliacao/juiz.py` (o juiz, com o Claude pelo Claude Code) |
| `revisao_manual.csv` | a revisão humana dos vereditos (o modelo é gerado na Parte 6) |
| `ambiente.json` | notebook 03 (hardware, modelo e versões) |

**Acerto** = veredito **correta**; parcial conta como erro. O acerto que vale (*acerto final*) é o da revisão humana quando a execução foi revisada, e o do juiz quando não foi. Os gráficos são gravados em `resultados/graficos/`, para o README e os slides.

In [ ]:
# @title Parte 1 · Carregar os resultados
import os, subprocess, sys

REPO_URL = "https://github.com/felipeafigueiredo/text-to-sql-and-cypher.git"
try:
    import google.colab                                         # no Colab: clona o repositório
    REPO_DIR = "/content/text-to-sql-and-cypher"
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
except ImportError:                                             # local: o notebook está em notebooks/
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, REPO_DIR)

RESULTADOS_DIR = os.path.join(REPO_DIR, "resultados")          # ou uma pasta do Drive com os mesmos arquivos
GRAFICOS_DIR = os.path.join(RESULTADOS_DIR, "graficos")

import pandas as pd
from IPython.display import Markdown, display
from avaliacao import analise as an
pd.set_option("display.max_colwidth", 120)

DADOS = an.carregar(RESULTADOS_DIR)
DF = an.por_execucao(DADOS)
print(f"{len(DF)} execuções · {DF['pergunta_id'].nunique()} perguntas · sistemas: {', '.join(DF['sistema'].unique().astype(str))}")
print(f"juiz: {'sim' if DADOS['juiz'] is not None else 'ainda não'} · "
      f"revisão humana: {0 if DADOS['revisao'] is None else len(DADOS['revisao'])} execuções")

---
## Parte 2 · Ambiente

O hardware, o modelo e as versões da rodada das 60 execuções (gravados pelo notebook 03).

In [ ]:
# @title Ambiente da rodada
amb = DADOS["ambiente"]
if amb:
    libs = amb.pop("bibliotecas", {})
    display(pd.Series({**amb, **{f"lib {k}": v for k, v in libs.items()}}, name="valor").to_frame())
else:
    print("ambiente.json não encontrado")

---
## Parte 3 · Tabela comparativa

Uma coluna por sistema, com as métricas obrigatórias (acerto, trajetória, latência média/mediana/p95, tokens de entrada e saída somados entre os agentes, chamadas ao modelo e às ferramentas, CPU, RAM e GPU) e as do tema: **consultas válidas** (consultas SQL e Cypher escritas pelo modelo que rodaram sem erro) e tentativas até a primeira consulta válida.

In [ ]:
# @title Tabela comparativa
TABELA = an.tabela_comparativa(DF)
TABELA.to_csv(os.path.join(RESULTADOS_DIR, "tabela_comparativa.csv"))
display(an.formatar(TABELA))

---
## Parte 4 · Acerto × custo

Cada sistema é um ponto. O custo aparece em dois painéis com a mesma escala de acerto (tokens por pergunta e latência mediana), e não num gráfico de dois eixos, porque as duas medidas têm escalas diferentes. Abaixo, a distribuição da latência de cada execução.

In [ ]:
# @title Acerto × custo e latência
an.grafico_acerto_custo(DF, GRAFICOS_DIR);
an.grafico_latencia(DF, GRAFICOS_DIR);

---
## Parte 5 · Erros por pergunta

Onde cada sistema errou: o veredito de cada execução, o acerto por categoria de pergunta e por banco esperado, e a lista das execuções que não foram consideradas corretas, com a justificativa do juiz.

In [ ]:
# @title Veredito por pergunta
an.grafico_mapa(DF, "veredito_juiz", GRAFICOS_DIR);
if DF["veredito_humano"].notna().any():
    an.grafico_mapa(DF, "veredito_humano", GRAFICOS_DIR, "vereditos_por_pergunta_humano.png");

In [ ]:
# @title Acerto por categoria e por banco
an.grafico_acerto_por(DF, "categoria", an.CATEGORIAS, "Acerto por categoria de pergunta", "acerto_por_categoria.png", GRAFICOS_DIR);
an.grafico_acerto_por(DF, "banco", an.BANCOS, "Acerto por banco esperado", "acerto_por_banco.png", GRAFICOS_DIR);
display(an.pct(an.acerto_por(DF, "categoria", an.CATEGORIAS)))
display(an.pct(an.acerto_por(DF, "banco", an.BANCOS)))

In [ ]:
# @title Execuções que não foram consideradas corretas
erros = DF[DF["acerto_final"] == False][["execucao", "categoria", "banco", "veredito_juiz", "veredito_humano",
                                          "checagem_falhas", "justificativa_juiz"]]
print(f"{len(erros)} de {len(DF)} execuções")
display(erros.set_index("execucao"))

---
## Parte 6 · Roteamento e verificador

**Sistema 2:** a rota escolhida pelo roteador contra o banco esperado da pergunta. Nas perguntas P11 e P12, que o SQL também responde, a rota `sql` conta como certa (T46).

**Sistema 3:** as respostas que o verificador devolveu, com o veredito do juiz para a primeira resposta do agente (antes) e para a final (depois).

In [ ]:
# @title Roteamento (sistema 2)
acerto_rota, matriz_rota = an.roteamento(DF)
if acerto_rota is not None:
    print("acerto de rota por banco esperado:")
    display(an.pct(acerto_rota.rename("rota certa")))
    print("rota esperada × rota escolhida:")
    display(matriz_rota)

In [ ]:
# @title Verificador (sistema 3)
dev = DF[DF["sistema"] == "verificador"]["devolucoes"].fillna(0)
print(f"devoluções: {int(dev.sum())} em {int((dev > 0).sum())} das {len(dev)} execuções do sistema 3")
mud = an.verificador(DADOS)
if mud is not None and len(mud):
    display(mud.set_index("execucao"))
    display(mud["efeito"].value_counts().to_frame("execuções"))

---
## Parte 7 · Revisão manual e concordância

A especificação pede a verificação manual dos vereditos. A célula abaixo cria `resultados/revisao_manual.csv` (se ainda não existir), com a pergunta, o gabarito, a resposta, a checagem automática e o veredito do juiz em cada linha. O grupo preenche a coluna **`veredito_humano`** com `correta`, `parcial` ou `incorreta` (e `observacao`, se quiser), sem olhar o veredito do juiz antes de decidir, se possível. Pode ser aberto no Excel ou no Google Planilhas: o separador `;` também é aceito. Depois, rode o notebook de novo.

Concordância em **acerto** (certo/errado) entre os três: humano, juiz e checagem automática, com o **kappa de Cohen** (concordância além do acaso: 1 é perfeita, 0 é a do acaso).

In [ ]:
# @title Modelo da revisão manual
caminho, criado = an.modelo_revisao(RESULTADOS_DIR, DADOS)
print(("criado: " if criado else "já existe (não foi sobrescrito): ") + caminho)

In [ ]:
# @title Concordância
pares, matriz = an.concordancia(DF)
display(pares.assign(concordância=pares["concordância"].map(lambda v: f"{v:.0%}"), kappa=pares["kappa"].round(2)))
if matriz is not None:
    print("vereditos: juiz (linhas) × humano (colunas)")
    display(matriz)
    print("execuções em que juiz e humano discordam no acerto:")
    display(an.discordancias(DF).set_index("execucao"))
else:
    print("ainda sem revisão humana: preencha veredito_humano em revisao_manual.csv e rode de novo")

In [ ]:
# @title Custo do juiz
c = an.custo_do_juiz(DADOS)
if c is not None:
    display(c.to_frame("juiz"))

---
## Parte 8 · Conclusão

Os números que a conclusão precisa, gerados dos resultados (para conferir: a conclusão em texto é escrita pelo grupo, a partir deles e da análise dos erros acima).

In [ ]:
# @title Fatos para a conclusão
for f in an.fatos(DF, DADOS):
    print("•", f)

### Conclusão

*(a escrever com os resultados da rodada final: qual sistema acertou mais e a que custo; onde a hipótese de cada arquitetura se confirmou ou não — o roteamento com menos contexto escreveu consultas melhores? o verificador pegou erros de sentido sem estragar respostas certas?; em que tipos de pergunta o modelo teve mais dificuldade; quanto o juiz concordou com a revisão humana.)*